In [ ]:
!pip install textstat

In [ ]:
import json
import pandas as pd
import numpy as np
import re
import spacy
from textstat import flesch_kincaid_grade
from nltk.corpus import stopwords
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

class Task2QueryAnalyzer:
    def __init__(self):
        # Setup
        nltk.download('stopwords', quiet=True)
        self.stop_words = set(stopwords.words('english'))
        self.nlp = spacy.load('en_core_web_sm')
        self.sentence_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    def load_data(self, json_path):
        """Load JSON data and return DataFrame"""
        with open(json_path, 'r') as f:
            data = json.load(f)

        # Extract task2_queries from the new schema
        queries = data["task2_queries"]
        df = pd.DataFrame(queries)
        return df

    # === LINGUISTIC VALIDITY METRICS ===
    def grammatical_completeness(self, doc):
        has_verb = any(tok.pos_ == "VERB" for tok in doc)
        has_noun = any(tok.pos_ == "NOUN" for tok in doc)
        return int(has_verb and has_noun)

    def interrogative_structure(self, text):
        pattern = re.compile(r"^(why|how|what|when|which|who)\b", re.IGNORECASE)
        return int(bool(pattern.search(text)) or text.strip().endswith("?"))

    def calculate_linguistic_validity(self, df):
        """Calculate linguistic validity metrics"""
        metrics = []

        for q in df["follow_up_query"]:
            doc = self.nlp(q)

            G = self.grammatical_completeness(doc)
            I = self.interrogative_structure(q)
            R_raw = flesch_kincaid_grade(q)

            metrics.append({
                "follow_up_query": q,
                "grammatical_completeness": G,
                "interrogative_structure": I,
                "readability_score": R_raw
            })

        metrics_df = pd.DataFrame(metrics)
        result = pd.concat([df, metrics_df.drop(columns=["follow_up_query"])], axis=1)
        return result

    # === READABILITY BY DIFFICULTY ===
    def calculate_readability_by_difficulty(self, df):
        """Calculate readability scores by difficulty level"""
        readability_by_difficulty = df.groupby("difficulty")["readability_score"].agg([
            'count', 'mean', 'std', 'min', 'max'
        ]).round(3)

        return readability_by_difficulty

    # === CORRECTED TF-IDF SIMILARITY - DOMAIN + FOLLOW_UP_TYPE ===
    def calculate_tfidf_similarity(self, df):
        """TF-IDF similarity based on domain + follow_up_type groups"""
        results_same_group = []  # Same domain + same follow_up_type
        results_inter_group = [] # Same domain + different follow_up_type

        # Generate TF-IDF matrix for all queries
        vectorizer = TfidfVectorizer(stop_words="english", lowercase=True, max_features=5000)
        tfidf_matrix = vectorizer.fit_transform(df["follow_up_query"])

        # Add index to dataframe for easy reference
        df = df.copy()
        df["query_index"] = range(len(df))

        # 1. INTRA-GROUP: Same Domain + Same Follow-up Type
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_follow_up_type = row["follow_up_type"]
            current_idx = row["query_index"]

            # Find other queries with same domain + same follow_up_type
            same_group_mask = (
                (df["domain"] == current_domain) &
                (df["follow_up_type"] == current_follow_up_type) &
                (df["query_index"] != current_idx)
            )
            same_group_indices = df[same_group_mask]["query_index"].tolist()

            if len(same_group_indices) > 0:
                # Calculate similarity with all other queries in same group
                similarities = []
                for other_idx in same_group_indices:
                    sim = cosine_similarity(
                        tfidf_matrix[current_idx:current_idx+1],
                        tfidf_matrix[other_idx:other_idx+1]
                    )[0][0]
                    similarities.append(sim)

                if similarities:
                    results_same_group.append({
                        "follow_up_query": row["follow_up_query"],
                        "domain": current_domain,
                        "follow_up_type": current_follow_up_type,
                        "intra_group_similarity": np.mean(similarities),
                        "comparisons_count": len(similarities),
                        "min_similarity": np.min(similarities),
                        "max_similarity": np.max(similarities)
                    })

        # 2. INTER-GROUP: Same Domain + Different Follow-up Type
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_follow_up_type = row["follow_up_type"]
            current_idx = row["query_index"]

            # Find queries with same domain but DIFFERENT follow-up types
            inter_group_mask = (
                (df["domain"] == current_domain) &
                (df["follow_up_type"] != current_follow_up_type)
            )
            inter_group_indices = df[inter_group_mask]["query_index"].tolist()

            if len(inter_group_indices) > 0:
                # Calculate similarity with queries from other follow-up types
                similarities = []
                for other_idx in inter_group_indices:
                    sim = cosine_similarity(
                        tfidf_matrix[current_idx:current_idx+1],
                        tfidf_matrix[other_idx:other_idx+1]
                    )[0][0]
                    similarities.append(sim)

                if similarities:
                    results_inter_group.append({
                        "follow_up_query": row["follow_up_query"],
                        "domain": current_domain,
                        "current_follow_up_type": current_follow_up_type,
                        "compared_follow_up_types": df[inter_group_mask]["follow_up_type"].nunique(),
                        "inter_group_similarity": np.mean(similarities),
                        "comparisons_count": len(similarities),
                        "min_similarity": np.min(similarities),
                        "max_similarity": np.max(similarities)
                    })

        return pd.DataFrame(results_same_group), pd.DataFrame(results_inter_group)

    # === CORRECTED SEMANTIC SIMILARITY - DOMAIN + FOLLOW_UP_TYPE ===
    def calculate_semantic_similarity(self, df):
        """Semantic similarity based on domain + follow_up_type groups"""
        results_intra_group = []  # Same domain + same follow_up_type
        results_inter_group = []  # Same domain + different follow_up_type

        print("Generating embeddings...")

        # Generate embeddings for all queries
        embeddings = self.sentence_model.encode(
            df["follow_up_query"].tolist(),
            normalize_embeddings=False,
            show_progress_bar=False
        )

        df = df.copy()
        df["embedding"] = list(embeddings)
        df["query_index"] = range(len(df))

        # 1. INTRA-GROUP: Same Domain + Same Follow-up Type
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_follow_up_type = row["follow_up_type"]
            current_embedding = row["embedding"]
            current_idx = row["query_index"]

            # Find other queries with same domain + same follow_up_type
            intra_group_mask = (
                (df["domain"] == current_domain) &
                (df["follow_up_type"] == current_follow_up_type) &
                (df["query_index"] != current_idx)
            )
            intra_group_df = df[intra_group_mask]

            if len(intra_group_df) > 0:
                # Calculate similarity with all other queries in same group
                similarities = []
                for _, other_row in intra_group_df.iterrows():
                    other_embedding = other_row["embedding"]
                    # Manual cosine similarity
                    dot_product = np.dot(current_embedding, other_embedding)
                    norm1 = np.linalg.norm(current_embedding)
                    norm2 = np.linalg.norm(other_embedding)

                    if norm1 > 0 and norm2 > 0:
                        cos_sim = dot_product / (norm1 * norm2)
                        similarities.append(cos_sim)

                if similarities:
                    results_intra_group.append({
                        "follow_up_query": row["follow_up_query"],
                        "domain": current_domain,
                        "follow_up_type": current_follow_up_type,
                        "intra_group_semantic_similarity": np.mean(similarities),
                        "semantic_comparisons_count": len(similarities),
                        "min_semantic_similarity": np.min(similarities),
                        "max_semantic_similarity": np.max(similarities)
                    })

        # 2. INTER-GROUP: Same Domain + Different Follow-up Type
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_follow_up_type = row["follow_up_type"]
            current_embedding = row["embedding"]

            # Find queries with same domain but DIFFERENT follow-up types
            inter_group_mask = (
                (df["domain"] == current_domain) &
                (df["follow_up_type"] != current_follow_up_type)
            )
            inter_group_df = df[inter_group_mask]

            if len(inter_group_df) > 0:
                # Calculate similarity with queries from other follow-up types
                similarities = []
                for _, other_row in inter_group_df.iterrows():
                    other_embedding = other_row["embedding"]
                    # Manual cosine similarity
                    dot_product = np.dot(current_embedding, other_embedding)
                    norm1 = np.linalg.norm(current_embedding)
                    norm2 = np.linalg.norm(other_embedding)

                    if norm1 > 0 and norm2 > 0:
                        cos_sim = dot_product / (norm1 * norm2)
                        similarities.append(cos_sim)

                if similarities:
                    results_inter_group.append({
                        "follow_up_query": row["follow_up_query"],
                        "domain": current_domain,
                        "current_follow_up_type": current_follow_up_type,
                        "compared_follow_up_types": inter_group_df["follow_up_type"].nunique(),
                        "inter_group_semantic_similarity": np.mean(similarities),
                        "semantic_comparisons_count": len(similarities),
                        "min_inter_similarity": np.min(similarities),
                        "max_inter_similarity": np.max(similarities)
                    })

        return pd.DataFrame(results_intra_group), pd.DataFrame(results_inter_group)

    # === CONTEXT DEPENDENCY ANALYSIS ===
    def calculate_context_dependency(self, df):
        """Analyze context dependency patterns"""
        context_analysis = df.groupby(["difficulty", "context_dependency"]).agg({
            "follow_up_query": "count",
            "readability_score": "mean"
        }).round(3).reset_index()

        context_analysis = context_analysis.rename(columns={"follow_up_query": "query_count"})
        return context_analysis

    # === ANALYTICAL PROGRESSION ANALYSIS ===
    def calculate_analytical_progression(self, df):
        """Analyze analytical progression patterns"""
        progression_analysis = df.groupby(["difficulty", "analytical_progression"]).agg({
            "follow_up_query": "count",
            "readability_score": "mean"
        }).round(3).reset_index()

        progression_analysis = progression_analysis.rename(columns={"follow_up_query": "query_count"})
        return progression_analysis

    # === DOMAIN + FOLLOW_UP_TYPE DISTRIBUTION ===
    def calculate_domain_followup_distribution(self, df):
        """Analyze distribution across domains and follow-up types"""
        distribution = df.groupby(["domain", "follow_up_type", "difficulty"]).agg({
            "follow_up_query": "count",
            "readability_score": "mean"
        }).round(3).reset_index()

        distribution = distribution.rename(columns={"follow_up_query": "query_count"})
        return distribution

    # === COMPREHENSIVE ANALYSIS ===
    def analyze_queries(self, json_path):
        """Run complete analysis with correct methodology for Task 2 queries"""
        print(f"Loading data from {json_path}...")
        df = self.load_data(json_path)

        print(f"Total Task 2 queries: {len(df)}")
        print(f"Unique domains: {df['domain'].nunique()}")
        print(f"Domains: {df['domain'].unique().tolist()}")
        print(f"Follow-up types: {df['follow_up_type'].unique().tolist()}")
        print(f"Difficulties: {df['difficulty'].unique().tolist()}")

        print("\n=== LINGUISTIC VALIDITY ANALYSIS ===")
        linguistic_df = self.calculate_linguistic_validity(df)

        grammar_ok = linguistic_df["grammatical_completeness"].mean() * 100
        interrogative_ok = linguistic_df["interrogative_structure"].mean() * 100
        print(f"Grammar OK: {grammar_ok:.1f}%")
        print(f"Interrogative OK: {interrogative_ok:.1f}%")

        print("\n=== READABILITY BY DIFFICULTY LEVEL ===")
        readability_by_difficulty = self.calculate_readability_by_difficulty(linguistic_df)
        print(readability_by_difficulty)

        print("\n=== DOMAIN + FOLLOW-UP TYPE DISTRIBUTION ===")
        distribution = self.calculate_domain_followup_distribution(linguistic_df)
        print(distribution)

        print("\n=== CONTEXT DEPENDENCY ANALYSIS ===")
        context_analysis = self.calculate_context_dependency(linguistic_df)
        print(context_analysis)

        print("\n=== ANALYTICAL PROGRESSION ANALYSIS ===")
        progression_analysis = self.calculate_analytical_progression(linguistic_df)
        print(progression_analysis)

        print("\n=== TF-IDF SIMILARITY (Domain + Follow-up Type) ===")
        tfidf_intra, tfidf_inter = self.calculate_tfidf_similarity(linguistic_df)

        print("INTRA-GROUP: Same Domain + Same Follow-up Type:")
        if len(tfidf_intra) > 0:
            print(f"  Queries analyzed: {len(tfidf_intra)}")
            print(f"  Avg Similarity: {tfidf_intra['intra_group_similarity'].mean():.3f}")
            print(f"  Range: {tfidf_intra['min_similarity'].mean():.3f} - {tfidf_intra['max_similarity'].mean():.3f}")

        print("INTER-GROUP: Same Domain + Different Follow-up Type:")
        if len(tfidf_inter) > 0:
            print(f"  Queries analyzed: {len(tfidf_inter)}")
            print(f"  Avg Similarity: {tfidf_inter['inter_group_similarity'].mean():.3f}")
            print(f"  Range: {tfidf_inter['min_similarity'].mean():.3f} - {tfidf_inter['max_similarity'].mean():.3f}")

        # Calculate similarity gap
        if len(tfidf_intra) > 0 and len(tfidf_inter) > 0:
            intra_avg = tfidf_intra['intra_group_similarity'].mean()
            inter_avg = tfidf_inter['inter_group_similarity'].mean()
            similarity_gap = intra_avg - inter_avg
            print(f"  Similarity Gap (Intra - Inter): {similarity_gap:.3f}")

        print("\n=== SEMANTIC SIMILARITY (Domain + Follow-up Type) ===")
        semantic_intra, semantic_inter = self.calculate_semantic_similarity(linguistic_df)

        print("INTRA-GROUP: Same Domain + Same Follow-up Type:")
        if len(semantic_intra) > 0:
            print(f"  Queries analyzed: {len(semantic_intra)}")
            print(f"  Avg Semantic Similarity: {semantic_intra['intra_group_semantic_similarity'].mean():.3f}")
            print(f"  Range: {semantic_intra['min_semantic_similarity'].mean():.3f} - {semantic_intra['max_semantic_similarity'].mean():.3f}")

        print("INTER-GROUP: Same Domain + Different Follow-up Type:")
        if len(semantic_inter) > 0:
            print(f"  Queries analyzed: {len(semantic_inter)}")
            print(f"  Avg Semantic Similarity: {semantic_inter['inter_group_semantic_similarity'].mean():.3f}")
            print(f"  Range: {semantic_inter['min_inter_similarity'].mean():.3f} - {semantic_inter['max_inter_similarity'].mean():.3f}")

        # Calculate semantic similarity gap
        if len(semantic_intra) > 0 and len(semantic_inter) > 0:
            intra_avg = semantic_intra['intra_group_semantic_similarity'].mean()
            inter_avg = semantic_inter['inter_group_semantic_similarity'].mean()
            semantic_gap = intra_avg - inter_avg
            print(f"  Semantic Similarity Gap (Intra - Inter): {semantic_gap:.3f}")

        # Save results
        self.save_results(linguistic_df, readability_by_difficulty, distribution,
                         context_analysis, progression_analysis,
                         tfidf_intra, tfidf_inter, semantic_intra, semantic_inter)

        return linguistic_df

    def save_results(self, linguistic_df, readability_by_difficulty, distribution,
                    context_analysis, progression_analysis,
                    tfidf_intra, tfidf_inter, semantic_intra, semantic_inter):
        """Save all results to CSV files"""
        linguistic_df.to_csv("task2_linguistic_validity_analysis.csv", index=False)
        readability_by_difficulty.to_csv("task2_readability_by_difficulty.csv")
        distribution.to_csv("task2_domain_followup_distribution.csv", index=False)
        context_analysis.to_csv("task2_context_dependency_analysis.csv", index=False)
        progression_analysis.to_csv("task2_analytical_progression_analysis.csv", index=False)
        tfidf_intra.to_csv("task2_tfidf_intra_group_similarity.csv", index=False)
        tfidf_inter.to_csv("task2_tfidf_inter_group_similarity.csv", index=False)
        semantic_intra.to_csv("task2_semantic_intra_group_similarity.csv", index=False)
        semantic_inter.to_csv("task2_semantic_inter_group_similarity.csv", index=False)

        print("\n Task 2 Results saved:")
        print("  - task2_linguistic_validity_analysis.csv")
        print("  - task2_readability_by_difficulty.csv")
        print("  - task2_domain_followup_distribution.csv")
        print("  - task2_context_dependency_analysis.csv")
        print("  - task2_analytical_progression_analysis.csv")
        print("  - task2_tfidf_intra_group_similarity.csv")
        print("  - task2_tfidf_inter_group_similarity.csv")
        print("  - task2_semantic_intra_group_similarity.csv")
        print("  - task2_semantic_inter_group_similarity.csv")

# === USAGE ===
if __name__ == "__main__":
    analyzer = Task2QueryAnalyzer()

    YOUR_JSON_FILE = "sorted_queries.json"  # ← CHANGE THIS

    try:
        results = analyzer.analyze_queries(YOUR_JSON_FILE)
        print("\n Analysis completed successfully!")

    except FileNotFoundError:
        print(f"\n Error: File '{YOUR_JSON_FILE}' not found!")
    except Exception as e:
        print(f"\n An error occurred: {e}")
        import traceback
        traceback.print_exc()